# Pseudo-pairing result analysis pipeline

Stage 1 aggregates seed-level evaluation outputs and creates variant-selection heatmaps. Stage 2 plots final selected comparisons.

In [1]:
from pathlib import Path
from types import SimpleNamespace

from result_analysis_common import build_default_task_inputs, FINAL_METRICS_BY_TASK
from run_result_analysis_pipeline import run_result_analysis_pipeline

In [2]:
DATASET_ID = "Replogle_RPE"
PERTURBED_GROUP = "single"  # single, dual, or multi

EVAL_ROOT = Path("/ibex/user/chenj0i/Perturbation/evaluation/Replogle_RPE_pseudo_pairing_evaluation")
OUTDIR = EVAL_ROOT / PERTURBED_GROUP / "result_analysis"
OUTDIR.mkdir(parents=True, exist_ok=True)

MLP_CONFIG_SUMMARY_PATH = (EVAL_ROOT / PERTURBED_GROUP / "downstream_mlp" / "mlp_config_summary.json")

TASK_INPUTS = build_default_task_inputs(
    eval_root=EVAL_ROOT,
    perturbed_group=PERTURBED_GROUP,
    tasks=["control_manifold", "perturbation_effect", "mlp_forward", "mlp_inverse"],
    allow_missing=True,
)
TASK_INPUTS

{'control_manifold': {'input_path': PosixPath('/ibex/user/chenj0i/Perturbation/evaluation/Replogle_RPE_pseudo_pairing_evaluation/single/control_manifold/control_manifold_preservation_repeated_long.csv'),
  'strategy_col': 'strategy',
  'metrics': ['mean_expression_rmse',
   'mean_expression_correlation',
   'variance_pearson',
   'control_pseudo_local_mixing_score',
   'pca_centroid_distance',
   'mmd_pca']},
 'perturbation_effect': {'input_path': PosixPath('/ibex/user/chenj0i/Perturbation/evaluation/Replogle_RPE_pseudo_pairing_evaluation/single/perturbation_effect/perturbation_effect_consistency_repeated_run_summary.csv'),
  'strategy_col': 'strategy',
  'metrics': ['perturbation_effect_rmse',
   'perturbation_effect_pearson',
   'perturbation_effect_magnitude_ratio',
   'top100_perturbation_effect_correlation']},
 'mlp_forward': {'input_path': PosixPath('/ibex/user/chenj0i/Perturbation/evaluation/Replogle_RPE_pseudo_pairing_evaluation/single/downstream_mlp/forward_mlp_run_summary.csv

In [3]:
import json
with open(MLP_CONFIG_SUMMARY_PATH, 'rb') as f:
    config_info = json.load(f)
perturb_calsses = config_info["n_perturbation_classes"]

## Stage 1: aggregate over random seeds only

This writes one row per strategy variant. A complete S0-S5 table should have S0=1, S1=1, S2=1, S3=15, S4=5, S5=15 rows.

In [4]:
CONFIG = SimpleNamespace(
    dataset_id=DATASET_ID,
    perturbed_group=PERTURBED_GROUP,
    eval_root=EVAL_ROOT,
    outdir=OUTDIR,
    task_inputs=TASK_INPUTS,

    run_aggregation=True,
    run_final_comparison=False,

    allow_missing_tasks=True,
    allow_task_failures=False,
    strict_metrics=True,

    final_metrics_by_task=FINAL_METRICS_BY_TASK,
    plot_s0_reference=True,
    plot_s1_reference=True,
    # Draw inverse naive baseline only for test_accuracy: 1 / n_perturbation_classes.
    # Precision/recall/macro F1 do not receive automatic 1/n baselines.
    plot_naive_inverse_reference=True,
    mlp_config_summary_path=MLP_CONFIG_SUMMARY_PATH,
    mlp_inverse_n_classes=perturb_calsses,

    # Optional: supply empirical dummy-classifier baselines for inverse precision,
    # recall, or macro F1 only if you compute them separately.
    inverse_metric_reference_values={},
    inverse_metric_reference_labels={},
)

outputs = run_result_analysis_pipeline(CONFIG)
outputs


[Aggregate] control_manifold
[Saved] /ibex/user/chenj0i/Perturbation/evaluation/Replogle_RPE_pseudo_pairing_evaluation/single/result_analysis/aggregated_by_task/control_manifold/control_manifold_seed_averaged_by_strategy_variant_wide.csv
[Variant counts]
strategy
S0_naive_mean_control_reference       1
S1_random_single_control              1
S2_random_average_controls            1
S3_SEACell_metacell_average          15
S4_SEACell_balanced_random_sample     5
S5_SEACell_OT_sampled_average        15

[Aggregate] perturbation_effect
[Saved] /ibex/user/chenj0i/Perturbation/evaluation/Replogle_RPE_pseudo_pairing_evaluation/single/result_analysis/aggregated_by_task/perturbation_effect/perturbation_effect_seed_averaged_by_strategy_variant_wide.csv
[Variant counts]
strategy
S0_naive_mean_control_reference       1
S1_random_single_control              1
S2_random_average_controls            1
S3_SEACell_metacell_average          15
S4_SEACell_balanced_random_sample     5
S5_SEACell_OT_sampled

{'aggregation': {'aggregation_manifest': PosixPath('/ibex/user/chenj0i/Perturbation/evaluation/Replogle_RPE_pseudo_pairing_evaluation/single/result_analysis/aggregated_by_task/aggregation_outputs_manifest.csv'),
  'selection_template': PosixPath('/ibex/user/chenj0i/Perturbation/evaluation/Replogle_RPE_pseudo_pairing_evaluation/single/result_analysis/selected_variants_TEMPLATE_EDIT_ME.csv'),
  'outputs': {'control_manifold': {'task_name': 'control_manifold',
    'task_dir': PosixPath('/ibex/user/chenj0i/Perturbation/evaluation/Replogle_RPE_pseudo_pairing_evaluation/single/result_analysis/aggregated_by_task/control_manifold'),
    'canonical_path': PosixPath('/ibex/user/chenj0i/Perturbation/evaluation/Replogle_RPE_pseudo_pairing_evaluation/single/result_analysis/aggregated_by_task/control_manifold/control_manifold_canonical_input_with_required_metrics.csv'),
    'summary_wide_path': PosixPath('/ibex/user/chenj0i/Perturbation/evaluation/Replogle_RPE_pseudo_pairing_evaluation/single/result

Edit `selected_variants_TEMPLATE_EDIT_ME.csv` and set `select_for_final=True` for the variants to use in final comparison.

In [5]:
selection_path = OUTDIR / "selected_variants_TEMPLATE_EDIT_ME.csv"
print(selection_path)

/ibex/user/chenj0i/Perturbation/evaluation/Replogle_RPE_pseudo_pairing_evaluation/single/result_analysis/selected_variants_TEMPLATE_EDIT_ME.csv


## Stage 2: final selected comparison

In [11]:
# Run after editing selected_variants_TEMPLATE_EDIT_ME.csv
CONFIG.run_aggregation = False
CONFIG.run_final_comparison = True
CONFIG.selection_path = OUTDIR / "selected_variants_TEMPLATE_EDIT_ME.csv"

final_outputs = run_result_analysis_pipeline(CONFIG)
final_outputs


[Final comparison] control_manifold
[Saved] /ibex/user/chenj0i/Perturbation/evaluation/Replogle_RPE_pseudo_pairing_evaluation/single/result_analysis/final_selected_comparison/control_manifold/control_manifold_selected_final_compare.csv
[Saved] 6 figures to /ibex/user/chenj0i/Perturbation/evaluation/Replogle_RPE_pseudo_pairing_evaluation/single/result_analysis/final_selected_comparison/control_manifold/figures

[Final comparison] perturbation_effect
[Saved] /ibex/user/chenj0i/Perturbation/evaluation/Replogle_RPE_pseudo_pairing_evaluation/single/result_analysis/final_selected_comparison/perturbation_effect/perturbation_effect_selected_final_compare.csv
[Saved] 4 figures to /ibex/user/chenj0i/Perturbation/evaluation/Replogle_RPE_pseudo_pairing_evaluation/single/result_analysis/final_selected_comparison/perturbation_effect/figures

[Final comparison] mlp_forward
[Saved] /ibex/user/chenj0i/Perturbation/evaluation/Replogle_RPE_pseudo_pairing_evaluation/single/result_analysis/final_selected_

{'final_comparison': {'final_manifest': PosixPath('/ibex/user/chenj0i/Perturbation/evaluation/Replogle_RPE_pseudo_pairing_evaluation/single/result_analysis/final_selected_comparison/final_comparison_outputs_manifest.csv'),
  'records': [{'task_name': 'control_manifold',
    'selected_table': '/ibex/user/chenj0i/Perturbation/evaluation/Replogle_RPE_pseudo_pairing_evaluation/single/result_analysis/final_selected_comparison/control_manifold/control_manifold_selected_final_compare.csv',
    'figure_dir': '/ibex/user/chenj0i/Perturbation/evaluation/Replogle_RPE_pseudo_pairing_evaluation/single/result_analysis/final_selected_comparison/control_manifold/figures',
    'n_figures': 6},
   {'task_name': 'perturbation_effect',
    'selected_table': '/ibex/user/chenj0i/Perturbation/evaluation/Replogle_RPE_pseudo_pairing_evaluation/single/result_analysis/final_selected_comparison/perturbation_effect/perturbation_effect_selected_final_compare.csv',
    'figure_dir': '/ibex/user/chenj0i/Perturbation/